# Bank account management

This lab models bank accounts as objects. A procedural version, made of dictionaries and functions, is given: read it and run it, then rewrite it as objects, exercise after exercise. Each exercise builds on the previous one: if you get stuck, run its solution before moving on to the next. Amounts are whole numbers of euros.

## Specifications

- There are four types of accounts:
  - the current account, which allows an overdraft of at most €500;
  - the joint account, a current account with two holders;
  - the savings account, whose balance can never be negative;
  - the Livret A, a savings account whose balance can't exceed its ceiling of €22,950.
- Anyone can deposit money into an account; only one of its holders can withdraw from it, or transfer from it to another account.
- A transfer only happens if both accounts accept it; otherwise, neither balance changes.
- A customer has at most one account of each type. A joint account counts for both its holders.
- A refused operation raises an exception that says why.

## The Procedural Version

The following code follows these rules with dictionaries and functions, except the last rule about customers, which it ignores. Run it, then the next cell, which plays a scenario.

In [ ]:
from typing import Any

OVERDRAFT_LIMIT = 500
LIVRET_A_CEILING = 22_950


def create_account(account_type: str, *holders: str) -> dict[str, Any]:
  return {"type": account_type, "holders": holders, "balance": 0}


def check_balance(account: dict[str, Any], new_balance: int) -> None:
  if account["type"] in ("current", "joint") and new_balance < -OVERDRAFT_LIMIT:
    raise ValueError(f"Overdraft exceeded: the balance would become €{new_balance}")
  if account["type"] in ("savings", "livret A") and new_balance < 0:
    raise ValueError(f"Insufficient funds: the balance would become €{new_balance}")
  if account["type"] == "livret A" and new_balance > LIVRET_A_CEILING:
    raise ValueError(f"Ceiling exceeded: the balance would become €{new_balance}")


def check_holder(account: dict[str, Any], requester: str) -> None:
  if requester not in account["holders"]:
    raise ValueError(f"{requester} isn't a holder of this account")


def deposit(account: dict[str, Any], amount: int) -> None:
  check_balance(account, account["balance"] + amount)
  account["balance"] += amount


def withdraw(account: dict[str, Any], requester: str, amount: int) -> None:
  check_holder(account, requester)
  check_balance(account, account["balance"] - amount)
  account["balance"] -= amount


def transfer(
  source: dict[str, Any], requester: str, target: dict[str, Any], amount: int
) -> None:
  check_holder(source, requester)
  check_balance(source, source["balance"] - amount)
  check_balance(target, target["balance"] + amount)
  source["balance"] -= amount
  target["balance"] += amount

In [ ]:
current = create_account("current", "Isabelle")
savings = create_account("savings", "Jean")
livret = create_account("livret A", "Jean")
joint = create_account("joint", "Isabelle", "Jean")
deposit(current, 1300)
deposit(savings, 1000)
deposit(livret, 1200)
deposit(joint, 6500)
transfer(current, "Isabelle", joint, 500)
withdraw(livret, "Jean", 60)
print(current["balance"], savings["balance"], livret["balance"], joint["balance"])

try:
  withdraw(savings, "Jean", 2000)
except ValueError as e:
  print(f"Refused: {e}")

savings["balance"] = -1_000_000  # Nothing prevents it
print(savings)

This code works, but:

- each new account type adds `if`s to `check_balance`, and a misspelled type (`"Livret A"`) is subject to no rule at all;
- nothing prevents modifying a balance directly, as the last line does;
- every error is a `ValueError`: the caller can't tell a refusal the rules provide for from a programming error, nor an exceeded overdraft from a reached ceiling;
- nothing links a customer to their accounts, and the one-account-of-each-type rule isn't enforced.

The following exercises rewrite this code as objects, which settle each of these points.

## Business Exceptions

Write a hierarchy of exceptions for refused operations:

- `BusinessError`, which inherits from `Exception`, the base class of all the others;
- `InsufficientFundsError`: the balance would become negative, or exceed the allowed overdraft;
- `CeilingExceededError`: the balance would exceed the account's ceiling;
- `AccessDeniedError`: the requester isn't a holder of the account;
- `DuplicateAccountError`: the customer already has an account of this type.

Like the standard library's exceptions, their names end with `Error`. Check that an `except BusinessError` catches a `CeilingExceededError`, and print its message.

In [ ]:
# Your code here

### Additional hints

A custom exception is a class that inherits from another exception; its body can be just a docstring. The message is the first argument passed to the exception (`raise CeilingExceededError("...")`), and `str(e)` returns it. An `except` catches the exception of the class it names and those of all its subclasses.

```python
class BusinessError(Exception):
  """Operation refused by the bank's rules."""


class InsufficientFundsError(...):
  """..."""


...

try:
  raise CeilingExceededError("Ceiling of €22,950 exceeded")
except BusinessError as e:
  print(f"Refused: {e}")
```

### Solution

In [ ]:
class BusinessError(Exception):
  """Operation refused by the bank's rules."""


class InsufficientFundsError(BusinessError):
  """The balance would become negative, or exceed the allowed overdraft."""


class CeilingExceededError(BusinessError):
  """The balance would exceed the account's ceiling."""


class AccessDeniedError(BusinessError):
  """The requester isn't a holder of the account."""


class DuplicateAccountError(BusinessError):
  """The customer already has an account of this type."""


try:
  raise CeilingExceededError("Ceiling of €22,950 exceeded")
except BusinessError as e:
  print(f"Refused: {type(e).__name__}: {e}")

## The `Account` Abstract Class

Write an abstract class `Account`, the base class of every account:

- `__init__` receives the holder's name, stores the tuple of holders (`("Isabelle",)`) in the `holders` attribute, and a balance of 0 in the `_balance` attribute;
- the `balance` property gives the balance, read-only;
- the abstract method `_check_balance(new_balance)` raises a `BusinessError` if the account can't have this balance;
- `deposit(amount)`, `withdraw(requester, amount)` and `transfer(requester, target, amount)` perform the operations of the specifications, raising an `AccessDeniedError` when the requester isn't a holder;
- `__repr__` prints, for example, `CurrentAccount('Isabelle', balance=800)`.

Check that creating an `Account` raises a `TypeError`: the operations will be tested with the subclasses of the next exercise.

In [ ]:
# Your code here

### Additional hints

A property without a setter is read-only: `account.balance = 0` raises an `AttributeError`, while the class's methods modify `self._balance`. The operations check everything before modifying anything: `transfer` calls the source's `_check_balance` then the target's, and only then touches the balances. Each subclass will define its own rule in `_check_balance`, which `Account`'s operations call without knowing which type of account it is. A `_check_holder(requester)` method avoids writing the requester check twice. For `__repr__`, `", ".join(repr(h) for h in self.holders)` gives `'Isabelle', 'Jean'`.

```python
from abc import ABC, abstractmethod


class Account(ABC):
  def __init__(self, holder: str) -> None:
    self.holders = (holder,)
    self._balance = 0

  @property
  def balance(self) -> int:
    return ...

  @abstractmethod
  def _check_balance(self, new_balance: int) -> None: ...

  def _check_holder(self, requester: str) -> None:
    if ...:
      raise AccessDeniedError(f"{requester} isn't a holder of this account")

  def deposit(self, amount: int) -> None:
    self._check_balance(self._balance + amount)
    self._balance += amount

  def withdraw(self, requester: str, amount: int) -> None: ...

  def transfer(self, requester: str, target: "Account", amount: int) -> None:
    self._check_holder(requester)
    self._check_balance(...)
    target._check_balance(...)
    ...

  def __repr__(self) -> str:
    holders = ", ".join(repr(h) for h in self.holders)
    return f"{type(self).__name__}({holders}, balance={...})"


try:
  Account("Isabelle")
except TypeError as e:
  print(e)
```

### Solution

In [ ]:
from abc import ABC, abstractmethod


class Account(ABC):
  def __init__(self, holder: str) -> None:
    self.holders = (holder,)
    self._balance = 0

  @property
  def balance(self) -> int:
    return self._balance

  @abstractmethod
  def _check_balance(self, new_balance: int) -> None:
    """Raise a BusinessError if the account can't have this balance."""

  def _check_holder(self, requester: str) -> None:
    if requester not in self.holders:
      raise AccessDeniedError(f"{requester} isn't a holder of this account")

  def deposit(self, amount: int) -> None:
    self._check_balance(self._balance + amount)
    self._balance += amount

  def withdraw(self, requester: str, amount: int) -> None:
    self._check_holder(requester)
    self._check_balance(self._balance - amount)
    self._balance -= amount

  def transfer(self, requester: str, target: "Account", amount: int) -> None:
    self._check_holder(requester)
    self._check_balance(self._balance - amount)
    target._check_balance(target._balance + amount)
    self._balance -= amount
    target._balance += amount

  def __repr__(self) -> str:
    holders = ", ".join(repr(h) for h in self.holders)
    return f"{type(self).__name__}({holders}, balance={self._balance})"


try:
  Account("Isabelle")
except TypeError as e:
  print(f"Refused: {e}")

## Account Types

Write the four account types of the specifications, subclasses of `Account`:

- `CurrentAccount`, whose class attribute `overdraft_limit` is 500;
- `JointAccount`, a subclass of `CurrentAccount`, created with its two holders: `JointAccount("Isabelle", "Jean")`;
- `SavingsAccount`, whose balance can't be negative;
- `LivretA`, a subclass of `SavingsAccount`, whose class attribute `ceiling` is 22,950.

Each one also has a class attribute `label` (`"current account"`, `"joint account"`…), which the next exercise will use. A negative balance beyond the allowed overdraft raises an `InsufficientFundsError`, and a balance above the ceiling a `CeilingExceededError`.

Then test them with the check cell that follows the solution.

In [ ]:
# Your code here

### Additional hints

Each class only defines what sets it apart: its rule in `_check_balance`, its class attributes, and, for `JointAccount`, an `__init__` that calls `super().__init__(holder)` then adds the co-holder to the `holders`. `JointAccount` inherits `CurrentAccount`'s rule. `LivretA`'s `_check_balance` first calls `SavingsAccount`'s with `super()`, then checks the ceiling. In a method, `self.overdraft_limit` reads the class attribute.

```python
class CurrentAccount(Account):
  label = "current account"
  overdraft_limit = 500

  def _check_balance(self, new_balance: int) -> None:
    if new_balance < ...:
      raise InsufficientFundsError(
        f"Overdraft of €{self.overdraft_limit} exceeded: "
        f"the balance would become €{new_balance}"
      )


class JointAccount(CurrentAccount):
  label = "joint account"

  def __init__(self, holder: str, co_holder: str) -> None:
    super().__init__(holder)
    self.holders = ...


class SavingsAccount(Account):
  ...


class LivretA(SavingsAccount):
  label = "Livret A"
  ceiling = 22_950

  def _check_balance(self, new_balance: int) -> None:
    super()._check_balance(new_balance)
    ...
```

### Solution

In [ ]:
class CurrentAccount(Account):
  label = "current account"
  overdraft_limit = 500

  def _check_balance(self, new_balance: int) -> None:
    if new_balance < -self.overdraft_limit:
      raise InsufficientFundsError(
        f"Overdraft of €{self.overdraft_limit} exceeded: "
        f"the balance would become €{new_balance}"
      )


class JointAccount(CurrentAccount):
  label = "joint account"

  def __init__(self, holder: str, co_holder: str) -> None:
    super().__init__(holder)
    self.holders = (holder, co_holder)


class SavingsAccount(Account):
  label = "savings account"

  def _check_balance(self, new_balance: int) -> None:
    if new_balance < 0:
      raise InsufficientFundsError(
        f"Insufficient funds: the balance would become €{new_balance}"
      )


class LivretA(SavingsAccount):
  label = "Livret A"
  ceiling = 22_950

  def _check_balance(self, new_balance: int) -> None:
    super()._check_balance(new_balance)
    if new_balance > self.ceiling:
      raise CeilingExceededError(
        f"Ceiling of €{self.ceiling} exceeded: the balance would become €{new_balance}"
      )

### Check

The next cell replays the procedural version's scenario with your classes, then attempts operations the rules refuse. The `attempt` function runs an operation and prints whether it's accepted, or the business exception that refuses it.

Expected results: balances of €800, €1,000, €1,140 and €7,000; the 4th operation accepted, the others refused (`InsufficientFundsError`, `CeilingExceededError`, `AccessDeniedError`, then `InsufficientFundsError`: the overdraft would exceed €500); at the end, -€400 on the current account and still €7,000 on the joint account, which the refused transfer didn't touch; and an `AttributeError` when the balance is assigned.

In [ ]:
from collections.abc import Callable


def attempt(description: str, operation: Callable[[], None]) -> None:
  try:
    operation()
  except BusinessError as e:
    print(f"{description}: refused ({type(e).__name__}: {e})")
  else:
    print(f"{description}: accepted")


current = CurrentAccount("Isabelle")
savings = SavingsAccount("Jean")
livret = LivretA("Jean")
joint = JointAccount("Isabelle", "Jean")
current.deposit(1300)
savings.deposit(1000)
livret.deposit(1200)
joint.deposit(6500)
current.transfer("Isabelle", joint, 500)
livret.withdraw("Jean", 60)
print(current, savings, livret, joint, sep="\n")

attempt("Jean withdraws €2000 from his savings", lambda: savings.withdraw("Jean", 2000))
attempt("Jean deposits €22,000 into his Livret A", lambda: livret.deposit(22_000))
attempt("Jean withdraws €100 from Isabelle's", lambda: current.withdraw("Jean", 100))
attempt("Isabelle withdraws €1200", lambda: current.withdraw("Isabelle", 1200))
attempt(
  "Isabelle transfers €200 to joint", lambda: current.transfer("Isabelle", joint, 200)
)
print(current, joint, sep="\n")

try:
  current.balance = 1_000_000
except AttributeError as e:
  print(f"Refused: {e}")

## Customers

Write a `Customer` class, created with the customer's name, which associates a customer with their accounts:

- the `accounts` attribute is a dictionary, empty at first, whose keys are the account types (the classes themselves, such as `CurrentAccount`) and whose values are the customer's accounts;
- `add(account)` adds an account to the customer, raising an `AccessDeniedError` if the customer isn't one of its holders, and a `DuplicateAccountError` if they already have an account of this type;
- the `net_worth` property gives the sum of their accounts' balances;
- `__repr__` prints the customer's name and the list of their accounts.

An account exists without its customer, and a joint account belongs to two customers: this is an aggregation, not a composition. Test your class with the check cell that follows the solution.

In [ ]:
# Your code here

### Additional hints

`type(account)` returns an account's class, which serves as the key: `JointAccount` and `CurrentAccount` are two different keys, even though one inherits from the other. `in` tests whether a key is in a dictionary, and `sum` adds up the balances a generator expression over `self.accounts.values()` produces. The `DuplicateAccountError` message can use the account's `label` class attribute: `Isabelle already has a current account`.

```python
class Customer:
  def __init__(self, name: str) -> None:
    self.name = name
    self.accounts: dict[type[Account], Account] = {}

  def add(self, account: Account) -> None:
    if self.name not in account.holders:
      raise AccessDeniedError(...)
    if type(account) in self.accounts:
      raise DuplicateAccountError(f"{self.name} already has a {account.label}")
    self.accounts[...] = account

  @property
  def net_worth(self) -> int:
    return sum(...)

  def __repr__(self) -> str:
    return f"Customer({self.name!r}, accounts={list(...)})"
```

### Solution

In [ ]:
class Customer:
  def __init__(self, name: str) -> None:
    self.name = name
    self.accounts: dict[type[Account], Account] = {}

  def add(self, account: Account) -> None:
    if self.name not in account.holders:
      raise AccessDeniedError(f"{self.name} isn't a holder of {account!r}")
    if type(account) in self.accounts:
      raise DuplicateAccountError(f"{self.name} already has a {account.label}")
    self.accounts[type(account)] = account

  @property
  def net_worth(self) -> int:
    return sum(account.balance for account in self.accounts.values())

  def __repr__(self) -> str:
    return f"Customer({self.name!r}, accounts={list(self.accounts.values())})"

### Check

Expected results: Isabelle has the current account and the joint account, Jean the savings account, the Livret A and the joint account; their net worths are €6,600 and €9,140 (the joint account counts for both); Isabelle's second current account is refused (`DuplicateAccountError`), as is Jean's Livret A (`AccessDeniedError`), but not a Livret A in her name.

In [ ]:
isabelle = Customer("Isabelle")
jean = Customer("Jean")
isabelle.add(current)
isabelle.add(joint)
jean.add(savings)
jean.add(livret)
jean.add(joint)
print(isabelle, jean, sep="\n")
print(isabelle.net_worth, jean.net_worth)

attempt(
  "Isabelle opens a second current account",
  lambda: isabelle.add(CurrentAccount("Isabelle")),
)
attempt("Isabelle adds Jean's Livret A", lambda: isabelle.add(livret))
attempt("Isabelle opens a Livret A", lambda: isabelle.add(LivretA("Isabelle")))
print(isabelle.accounts[LivretA])